# LeetCode 758 — Bold Words in String bằng Python

**Link đề:** https://leetcode.com/problems/bold-words-in-string/
**Độ khó:** Easy theo danh mục của repo (LeetCode hiện ghi **Medium**) · **Chủ đề:** Array, Hash Table, String, Trie, String Matching

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def boldWords(self, words: list[str], s: str) -> str:
        ...
```

## Mục tiêu học
1. Hiểu yêu cầu: tô đậm **mọi lần xuất hiện** (kể cả chồng nhau) của các từ khóa, dùng **ít thẻ nhất** (gộp các đoạn chồng/chạm nhau).
2. Ôn `str.startswith(prefix, start)`, slicing, list `bool`, `''.join`.
3. Tách bài thành 2 pha: **đánh dấu** ký tự nào cần đậm → **dựng chuỗi kết quả** từ dấu.
4. Học 4 cách: mảng đánh dấu → **gộp khoảng** (merge intervals) → một lượt với biến `end` → **Trie**.
5. Trace, chứng minh, phân tích độ phức tạp, test đối chiếu, luyện biến thể (LeetCode 616, 56 Merge Intervals).

## 1. Đề bài

Cho mảng từ khóa `words` và chuỗi `s`. Hãy làm **đậm** mọi lần xuất hiện của từng từ khóa trong `s`: chèn thẻ `<b>` trước và `</b>` sau phần cần đậm.

Yêu cầu: dùng **số thẻ ít nhất có thể** — các phần đậm **chồng lên nhau** hoặc **nằm sát nhau** phải gộp thành **một** cặp thẻ. Kết quả phải là cách gắn thẻ hợp lệ (không lồng nhau, đóng mở đúng).

| words | s | Output | Giải thích |
|---|---|---|---|
| `["ab","bc"]` | `"aabcd"` | `"a<b>abc</b>d"` | `ab` ở [1,3), `bc` ở [2,4) chồng nhau → gộp [1,4) |
| `["ab","cb"]` | `"aabcd"` | `"a<b>ab</b>cd"` | `cb` không xuất hiện |

Ví dụ thêm (tự suy): `words = ["ab","cd"]`, `s = "abcd"` → `"<b>abcd</b>"` vì [0,2) và [2,4) **sát nhau** → gộp.

**Constraints (theo hiểu biết):** `1 <= len(s) <= 500`; `0 <= len(words) <= 50`; `1 <= len(words[i]) <= 10`; `s` và `words[i]` chỉ gồm chữ cái thường tiếng Anh. Bài này **giống hệt** LeetCode 616 *Add Bold Tag in String* (616 có constraints lớn hơn).

## 2. Nền tảng Python

### 2.1. Slicing và `str.startswith(prefix, start)`

- `s[i:j]` lấy đoạn từ `i` tới `j-1`.
- `s.startswith(w, i)` kiểm "đoạn bắt đầu tại chỉ số `i` có khớp `w` không" — **không tạo chuỗi con mới** như `s[i:i+len(w)] == w`.

In [1]:
s = "aabcd"
print(s[1:3], s[2:4])
print(s.startswith("ab", 1), s.startswith("ab", 0), s.startswith("bc", 2))
print(s.startswith("cd", 3), s.startswith("cde", 3))   # vượt cuối chuỗi -> False, không lỗi

ab bc
True False True
True False


### 2.2. `str.find(sub, start)` — tìm mọi lần xuất hiện, kể cả chồng nhau

`s.find(w, start)` trả chỉ số đầu tiên `>= start` mà `w` xuất hiện, hoặc `-1`. Để bắt cả lần xuất hiện **chồng nhau** (ví dụ `"aa"` trong `"aaa"` ở 0 và 1), lần tìm sau bắt đầu từ `vị_trí + 1`, **không phải** `vị_trí + len(w)`.

In [2]:
def all_occurrences(s, w):
    res = []
    i = s.find(w)
    while i != -1:
        res.append(i)
        i = s.find(w, i + 1)
    return res

print(all_occurrences("aaaa", "aa"))      # [0, 1, 2]
print("aaaa".count("aa"))                 # 2 -> count() KHÔNG đếm chồng nhau!

[0, 1, 2]
2


### 2.3. Nối chuỗi hiệu quả bằng list + `''.join`

Chuỗi trong Python **bất biến**; `res += "x"` trong vòng lặp có thể tạo chuỗi mới mỗi lần. Thói quen tốt: gom từng mảnh vào **list** rồi `''.join(list)` một lần ở cuối.

In [3]:
parts = []
for piece in ["a", "<b>", "abc", "</b>", "d"]:
    parts.append(piece)
print("".join(parts))

a<b>abc</b>d


### 2.4. Nửa khoảng `[start, end)`

Ta biểu diễn mỗi lần xuất hiện của từ `w` tại `i` bằng **nửa khoảng** `[i, i + len(w))` — lấy `i`, không lấy `i + len(w)`. Lợi ích: độ dài = `end - start`, và hai khoảng **sát nhau** khi `end` của khoảng trước **bằng** `start` của khoảng sau.

```text
s     : a  b  c  d
chỉ số: 0  1  2  3
"ab"  : [0, 2)
"cd"  : [2, 4)     -> sát nhau (2 == 2) -> gộp [0, 4)
```

## 3. Hai pha của lời giải

**Pha 1 — đánh dấu:** tạo `bold[i] = True` nếu ký tự `s[i]` nằm trong **ít nhất một** lần xuất hiện của một từ khóa.

**Pha 2 — dựng chuỗi:** đi qua `s`; mở `<b>` tại đầu mỗi **dãy liên tiếp** các ký tự `True`, đóng `</b>` tại cuối dãy đó.

Vì sao gộp tự động? Hai khoảng chồng hoặc sát nhau tạo ra một **dãy `True` liền mạch**, nên Pha 2 chỉ sinh **một** cặp thẻ cho cả dãy. Ví dụ `"aabcd"`:

```text
s    : a  a  b  c  d
bold : F  T  T  T  F      ->  a <b> a b c </b> d
```

## 4. Cách 1 — Mảng đánh dấu (trực giác)

Pha 1: với **mỗi vị trí** `i` và **mỗi từ** `w`, nếu `s.startswith(w, i)` thì đánh `True` cho `i .. i+len(w)-1`.

In [4]:
def mark_bold(words, s):
    n = len(s)
    bold = [False] * n
    for i in range(n):
        for w in words:
            if s.startswith(w, i):
                for k in range(i, i + len(w)):
                    bold[k] = True
    return bold

bold = mark_bold(["ab", "bc"], "aabcd")
print(list("aabcd"))
print(["T" if x else "F" for x in bold])

['a', 'a', 'b', 'c', 'd']
['F', 'T', 'T', 'T', 'F']


Pha 2: duyệt `i`; `bold[i]` là **đầu dãy** khi `i == 0` hoặc `bold[i-1]` là False; là **cuối dãy** khi `i == n-1` hoặc `bold[i+1]` là False.

In [5]:
def build_from_mask(s, bold):
    n = len(s)
    parts = []
    for i in range(n):
        if bold[i] and (i == 0 or not bold[i - 1]):
            parts.append("<b>")
        parts.append(s[i])
        if bold[i] and (i == n - 1 or not bold[i + 1]):
            parts.append("</b>")
    return "".join(parts)


def bold_mask(words, s):
    return build_from_mask(s, mark_bold(words, s))

print(bold_mask(["ab", "bc"], "aabcd"))
print(bold_mask(["ab", "cb"], "aabcd"))
print(bold_mask(["ab", "cd"], "abcd"))

a<b>abc</b>d
a<b>ab</b>cd
<b>abcd</b>


### Chi phí và nút thắt

Gọi `n = len(s)`, `W = len(words)`, `K` = độ dài từ tối đa. Mỗi cặp `(i, w)` tốn `O(K)` cho `startswith`, và khi khớp lại tốn `O(K)` để tô → **`O(n · W · K)`**. Với `500 · 50 · 10 = 2.5·10^5` — rất ổn.

Nhưng có **việc lặp**: khi nhiều từ khớp tại cùng `i` (ví dụ `"a"`, `"ab"`, `"abc"`), ta tô đi tô lại cùng ô. Thật ra tại mỗi `i` chỉ cần biết **từ dài nhất** khớp ở đó. Và các ô đã tô có thể được "nhớ" bằng một con số duy nhất: **vị trí xa nhất đã được tô tới** → Cách 3. Trước đó, xem góc nhìn "khoảng" (Cách 2) vì nó tổng quát nhất.

## 5. Cách 2 — Thu thập khoảng rồi gộp (Merge Intervals)

1. Thu mọi khoảng `[i, i + len(w))` mà `w` xuất hiện tại `i`.
2. Sắp theo `start`.
3. Gộp: duyệt lần lượt; nếu khoảng mới có `start <= end` của khoảng cuối đã gộp (**chồng hoặc sát**) → kéo dài `end = max(end, new_end)`; ngược lại mở khoảng mới.
4. Dựng chuỗi: chèn `<b>`/`</b>` tại biên các khoảng đã gộp.

In [6]:
def collect_intervals(words, s):
    intervals = []
    for w in words:
        for i in all_occurrences(s, w):
            intervals.append([i, i + len(w)])
    return intervals


def merge_intervals(intervals):
    merged = []
    for start, end in sorted(intervals):
        if merged and start <= merged[-1][1]:      # chồng hoặc SÁT nhau
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged

iv = collect_intervals(["ab", "bc"], "aabcd")
print("khoảng thô :", iv)
print("sau khi gộp:", merge_intervals(iv))
print(merge_intervals([[0, 2], [2, 4], [6, 7], [5, 6], [9, 10]]))

khoảng thô : [[1, 3], [2, 4]]
sau khi gộp: [[1, 4]]
[[0, 4], [5, 7], [9, 10]]


`sorted(intervals)` sắp list các `[start, end]` theo `start` trước, hòa thì theo `end` (so sánh list theo thứ tự từ điển). `merged[-1]` là khoảng cuối cùng đã gộp.

Điều kiện gộp là `start <= merged[-1][1]` (dấu `<=`), **không phải** `<`: dấu `=` xử lý trường hợp **sát nhau** mà đề yêu cầu gộp.

In [7]:
def bold_intervals(words, s):
    merged = merge_intervals(collect_intervals(words, s))
    parts, prev = [], 0
    for start, end in merged:
        parts.append(s[prev:start])
        parts.append("<b>" + s[start:end] + "</b>")
        prev = end
    parts.append(s[prev:])
    return "".join(parts)

print(bold_intervals(["ab", "bc"], "aabcd"))
print(bold_intervals(["ab", "cb"], "aabcd"))
print(bold_intervals(["ab", "cd"], "abcd"))

a<b>abc</b>d
a<b>ab</b>cd
<b>abcd</b>


## 6. Cách 3 — Một lượt với biến `end` (lời giải chính)

Ý tưởng: duyệt `i` từ trái sang phải, giữ `end` = **vị trí xa nhất (không tính) đã chắc chắn phải đậm** tính từ các từ bắt đầu ở `<= i`.

- Tại mỗi `i`, với mọi từ `w` khớp tại `i`: `end = max(end, i + len(w))`.
- Ký tự `s[i]` cần đậm ⟺ `i < end`.

Nhờ vậy ta không cần tô từng ô của mỗi từ — một phép `max` là đủ. Gộp với Pha 2 của Cách 1 để dựng chuỗi.

In [8]:
def bold_one_pass(words, s):
    n = len(s)
    bold = [False] * n
    end = 0
    for i in range(n):
        for w in words:
            if s.startswith(w, i):
                end = max(end, i + len(w))
        bold[i] = i < end
    return build_from_mask(s, bold)

print(bold_one_pass(["ab", "bc"], "aabcd"))
print(bold_one_pass(["ab", "cb"], "aabcd"))
print(bold_one_pass(["ab", "cd"], "abcd"))

a<b>abc</b>d
a<b>ab</b>cd
<b>abcd</b>


## 7. Trace từng bước

`words = ["ab","bc"]`, `s = "aabcd"`.

| i | s[i] | Từ khớp tại i | end (sau) | i < end? | bold |
|---:|:---:|---|---:|:---:|:---:|
| 0 | a | — | 0 | ❌ | F |
| 1 | a | `ab` → 1+2 = 3 | 3 | ✅ | T |
| 2 | b | `bc` → 2+2 = 4 | 4 | ✅ | T |
| 3 | c | — | 4 | ✅ | T |
| 4 | d | — | 4 | ❌ | F |

Pha 2: mở `<b>` trước `i = 1` (bold[0] = F), đóng `</b>` sau `i = 3` (bold[4] = F) → `"a<b>abc</b>d"`.

In [9]:
def trace_bold(words, s):
    n = len(s)
    bold = [False] * n
    end = 0
    print(f"words = {words}, s = {s!r}")
    for i in range(n):
        matched = [w for w in words if s.startswith(w, i)]
        for w in matched:
            end = max(end, i + len(w))
        bold[i] = i < end
        print(f"i={i} s[i]={s[i]!r} | khớp: {str(matched):<12} | end={end} | bold={bold[i]}")
    parts = []
    for i in range(n):
        opening = bold[i] and (i == 0 or not bold[i - 1])
        closing = bold[i] and (i == n - 1 or not bold[i + 1])
        if opening:
            parts.append("<b>")
        parts.append(s[i])
        if closing:
            parts.append("</b>")
        if opening or closing:
            print(f"  i={i}: {'mở <b> ' if opening else ''}{'đóng </b>' if closing else ''}")
    result = "".join(parts)
    print("=>", result)
    return result

trace_bold(["ab", "bc"], "aabcd")

words = ['ab', 'bc'], s = 'aabcd'
i=0 s[i]='a' | khớp: []           | end=0 | bold=False
i=1 s[i]='a' | khớp: ['ab']       | end=3 | bold=True
i=2 s[i]='b' | khớp: ['bc']       | end=4 | bold=True
i=3 s[i]='c' | khớp: []           | end=4 | bold=True
i=4 s[i]='d' | khớp: []           | end=4 | bold=False
  i=1: mở <b> 
  i=3: đóng </b>
=> a<b>abc</b>d


'a<b>abc</b>d'

### Trace nhiều đoạn, có đoạn sát nhau và từ lồng nhau

`words = ["aaa","aab","bc"]`, `s = "aaabbcc"`: `aaa` ở [0,3), `aab` ở [1,4), `bc` ở [4,6) → [0,3)∪[1,4) = [0,4), sát [4,6) → gộp [0,6) → `"<b>aaabbc</b>c"`.

In [10]:
trace_bold(["aaa", "aab", "bc"], "aaabbcc")
print()
trace_bold(["b", "dd"], "abcxyz")

words = ['aaa', 'aab', 'bc'], s = 'aaabbcc'
i=0 s[i]='a' | khớp: ['aaa']      | end=3 | bold=True
i=1 s[i]='a' | khớp: ['aab']      | end=4 | bold=True
i=2 s[i]='a' | khớp: []           | end=4 | bold=True
i=3 s[i]='b' | khớp: []           | end=4 | bold=True
i=4 s[i]='b' | khớp: ['bc']       | end=6 | bold=True
i=5 s[i]='c' | khớp: []           | end=6 | bold=True
i=6 s[i]='c' | khớp: []           | end=6 | bold=False
  i=0: mở <b> 
  i=5: đóng </b>
=> <b>aaabbc</b>c

words = ['b', 'dd'], s = 'abcxyz'
i=0 s[i]='a' | khớp: []           | end=0 | bold=False
i=1 s[i]='b' | khớp: ['b']        | end=2 | bold=True
i=2 s[i]='c' | khớp: []           | end=2 | bold=False
i=3 s[i]='x' | khớp: []           | end=2 | bold=False
i=4 s[i]='y' | khớp: []           | end=2 | bold=False
i=5 s[i]='z' | khớp: []           | end=2 | bold=False
  i=1: mở <b> đóng </b>
=> a<b>b</b>cxyz


'a<b>b</b>cxyz'

## 8. Cách 4 — Trie: tìm từ dài nhất khớp tại mỗi vị trí

Ở Cách 3, tại mỗi `i` ta thử **tất cả** `W` từ. **Trie** (cây tiền tố) gộp các từ có chung tiền tố: đi từ `s[i]` theo các nhánh, mỗi bước một ký tự; gặp nút đánh dấu "kết thúc từ" thì ghi nhận độ dài. Tối đa đi `K` bước → mỗi `i` tốn `O(K)` thay vì `O(W · K)`.

Cài Trie bằng dict lồng nhau: mỗi nút là một `dict` ký tự → nút con; khóa đặc biệt `"$"` đánh dấu kết thúc từ.

In [11]:
def build_trie(words):
    root = {}
    for w in words:
        node = root
        for ch in w:
            node = node.setdefault(ch, {})
        node["$"] = True
    return root

trie = build_trie(["ab", "abc", "bc"])
print(trie)

{'a': {'b': {'$': True, 'c': {'$': True}}}, 'b': {'c': {'$': True}}}


In [12]:
def longest_match_at(trie, s, i):
    node, best = trie, 0
    j = i
    while j < len(s) and s[j] in node:
        node = node[s[j]]
        j += 1
        if "$" in node:
            best = j - i
    return best


def bold_trie(words, s):
    trie = build_trie(words)
    n = len(s)
    bold = [False] * n
    end = 0
    for i in range(n):
        end = max(end, i + longest_match_at(trie, s, i))
        bold[i] = i < end
    return build_from_mask(s, bold)

print([longest_match_at(trie, "xabcd", i) for i in range(5)])
print(bold_trie(["ab", "bc"], "aabcd"))
print(bold_trie(["aaa", "aab", "bc"], "aaabbcc"))

[0, 3, 2, 0, 0]
a<b>abc</b>d
<b>aaabbc</b>c


`longest_match_at` trả 0 khi không có từ nào bắt đầu tại `i` → `end` không đổi. Với `"xabcd"`: tại `i = 1` khớp `ab` rồi `abc` → 3; tại `i = 2` khớp `bc` → 2.

## 9. Code nộp LeetCode

### Bản một lượt với `end` (khuyến nghị)

In [13]:
class Solution:
    def boldWords(self, words: list[str], s: str) -> str:
        n = len(s)
        bold = [False] * n
        end = 0
        for i in range(n):
            for w in words:
                if s.startswith(w, i):
                    end = max(end, i + len(w))
            bold[i] = i < end
        res = []
        for i in range(n):
            if bold[i] and (i == 0 or not bold[i - 1]):
                res.append("<b>")
            res.append(s[i])
            if bold[i] and (i == n - 1 or not bold[i + 1]):
                res.append("</b>")
        return "".join(res)


sol = Solution()
print(sol.boldWords(["ab", "bc"], "aabcd"))
print(sol.boldWords(["ab", "cb"], "aabcd"))

a<b>abc</b>d
a<b>ab</b>cd


### Bản gộp khoảng và bản Trie

Trên LeetCode đổi tên class thành `Solution`. (Lưu ý: tên tham số ở một số phiên bản đề là `words, S` — giữ đúng thứ tự `words` trước, chuỗi sau.)

In [14]:
class SolutionIntervals:
    def boldWords(self, words: list[str], s: str) -> str:
        intervals = sorted([i, i + len(w)] for w in words
                           for i in range(len(s)) if s.startswith(w, i))
        merged = []
        for a, c in intervals:
            if merged and a <= merged[-1][1]:
                merged[-1][1] = max(merged[-1][1], c)
            else:
                merged.append([a, c])
        out, prev = [], 0
        for a, c in merged:
            out += [s[prev:a], "<b>", s[a:c], "</b>"]
            prev = c
        out.append(s[prev:])
        return "".join(out)


class SolutionTrie:
    def boldWords(self, words: list[str], s: str) -> str:
        return bold_trie(words, s)


print(SolutionIntervals().boldWords(["ab", "cd"], "abcd"), SolutionTrie().boldWords(["ab", "cd"], "abcd"))

<b>abcd</b> <b>abcd</b>


`out += [a, b, c]` nối **nhiều phần tử** vào list (tương đương `out.extend([...])`).

## 10. Vì sao đúng?

**Pha 1 (Cách 3).** Gọi `B` = tập vị trí thuộc ít nhất một lần xuất hiện. Invariant: *sau khi xử lý `i`, `end = max{ j + len(w) : j <= i, w khớp tại j }` (hoặc 0).*
- Nếu `i ∈ B` thì có `j <= i`, `w` khớp tại `j` với `i < j + len(w) <= end` → `bold[i] = True`.
- Nếu `bold[i] = True` thì `i < end = j + len(w)` với `j <= i` → `i ∈ [j, j + len(w))` → `i ∈ B`.

Vậy `bold` mô tả **chính xác** `B`.

**Pha 2.** Mỗi **dãy liên tiếp tối đại** các vị trí `True` nhận đúng một `<b>` (ở đầu dãy) và một `</b>` (ở cuối dãy). Các thẻ không lồng nhau, đóng mở hợp lệ.

**Ít thẻ nhất.** Một cặp thẻ bao một đoạn **liên tục** mà mọi ký tự trong đó đều đậm, nên nó không thể phủ hai dãy tối đại khác nhau (giữa hai dãy luôn có ít nhất một ký tự không đậm). Mỗi dãy tối đại cần ít nhất một cặp thẻ → số cặp tối thiểu = số dãy tối đại = đúng số cặp ta sinh ra.

**Cách 2** tương đương: gộp khoảng chồng/sát chính là tạo các dãy tối đại.

## 11. Độ phức tạp

Gọi `n = len(s)`, `W = len(words)`, `K` = độ dài từ tối đa, `T = Σ len(w)` tổng độ dài các từ.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Mảng đánh dấu (tô từng ô) | `O(n · W · K)` | `O(n)` |
| 2. Gộp khoảng | `O(n · W · K + M log M)`, `M` = số lần xuất hiện ≤ `n · W` | `O(M)` |
| 3. Một lượt `end` | `O(n · W · K)` (bỏ phần tô lặp) | `O(n)` |
| 4. Trie | `O(T + n · K)` | `O(T)` cho Trie + `O(n)` |

Với constraints (`n ≤ 500`, `W ≤ 50`, `K ≤ 10`), cả 4 cách đều rất nhanh. Trie phát huy khi `W` lớn (nhiều từ khóa) — ý tưởng nền của thuật toán **Aho–Corasick** (có trong tag của bài) dùng cho tìm nhiều mẫu cùng lúc.

Output dài `O(n + số_thẻ)` = `O(n)`.

## 12. Bộ test

Ngoài các ví dụ, ta viết **oracle** độc lập: tạo tập `B` bằng cách duyệt mọi lần xuất hiện qua slicing, rồi dựng chuỗi bằng cách đi từng ký tự — cách viết khác hẳn để đối chiếu.

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual!r}")

sol = Solution()
check("đề 1 (chồng nhau)", sol.boldWords(["ab", "bc"], "aabcd"), "a<b>abc</b>d")
check("đề 2", sol.boldWords(["ab", "cb"], "aabcd"), "a<b>ab</b>cd")
check("sát nhau -> gộp", sol.boldWords(["ab", "cd"], "abcd"), "<b>abcd</b>")
check("không từ nào", sol.boldWords([], "abc"), "abc")
check("không khớp", sol.boldWords(["z"], "abc"), "abc")
check("toàn chuỗi", sol.boldWords(["abc"], "abc"), "<b>abc</b>")
check("xuất hiện chồng của cùng một từ", sol.boldWords(["aa"], "aaa"), "<b>aaa</b>")
check("hai đoạn tách rời", sol.boldWords(["a"], "aba"), "<b>a</b>b<b>a</b>")
check("từ lồng trong từ", sol.boldWords(["abc", "b"], "xabcx"), "x<b>abc</b>x")
check("nhiều đoạn", sol.boldWords(["aaa", "aab", "bc"], "aaabbcc"), "<b>aaabbc</b>c")

PASS  đề 1 (chồng nhau): 'a<b>abc</b>d'
PASS  đề 2: 'a<b>ab</b>cd'
PASS  sát nhau -> gộp: '<b>abcd</b>'
PASS  không từ nào: 'abc'
PASS  không khớp: 'abc'
PASS  toàn chuỗi: '<b>abc</b>'
PASS  xuất hiện chồng của cùng một từ: '<b>aaa</b>'
PASS  hai đoạn tách rời: '<b>a</b>b<b>a</b>'
PASS  từ lồng trong từ: 'x<b>abc</b>x'
PASS  nhiều đoạn: '<b>aaabbc</b>c'


In [16]:
import random

def oracle(words, s):
    B = set()
    for w in words:
        for i in range(len(s) - len(w) + 1):
            if s[i:i + len(w)] == w:
                B.update(range(i, i + len(w)))
    out = ""
    inside = False
    for i, ch in enumerate(s):
        if i in B and not inside:
            out += "<b>"; inside = True
        if i not in B and inside:
            out += "</b>"; inside = False
        out += ch
    if inside:
        out += "</b>"
    return out

rng = random.Random(758)
impls = [bold_mask, bold_intervals, bold_one_pass, bold_trie,
         Solution().boldWords, SolutionIntervals().boldWords, SolutionTrie().boldWords]
for _ in range(3000):
    s = "".join(rng.choice("abc") for _ in range(rng.randint(1, 12)))
    words = ["".join(rng.choice("abc") for _ in range(rng.randint(1, 3))) for _ in range(rng.randint(0, 4))]
    expect = oracle(words, s)
    for f in impls:
        assert f(words, s) == expect, (f, words, s)
print("3000 bộ ngẫu nhiên: 7 cài đặt khớp oracle")

3000 bộ ngẫu nhiên: 7 cài đặt khớp oracle


`B.update(range(...))` thêm nhiều phần tử vào set cùng lúc. Oracle dùng biến cờ `inside` để biết đang ở trong thẻ hay không — một cách dựng chuỗi khác với Pha 2, giúp phát hiện lỗi ở cả hai.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Thay thế trực tiếp bằng `str.replace` từng từ

In [17]:
def wrong_replace(words, s):
    for w in words:
        s = s.replace(w, "<b>" + w + "</b>")
    return s

print("sai :", wrong_replace(["ab", "bc"], "aabcd"), "| đúng:", bold_one_pass(["ab", "bc"], "aabcd"))
print("sai :", wrong_replace(["b", "b>"], "ab"))

sai : a<b>ab</b>cd | đúng: a<b>abc</b>d
sai : a<<b>b></b>b</<b>b></b>


Sai vì: (1) không gộp đoạn chồng nhau; (2) thay xong từ đầu thì chuỗi đã chứa thẻ, từ sau có thể khớp vào **chính các thẻ** (`"b"` khớp trong `"<b>"`!) → hỏng cấu trúc. Phải **đánh dấu trên chuỗi gốc** rồi dựng một lần.

### Lỗi 2: Gộp bằng `<` thay vì `<=` → không gộp đoạn sát nhau

In [18]:
def wrong_merge_strict(words, s):
    merged = []
    for a, c in sorted(collect_intervals(words, s)):
        if merged and a < merged[-1][1]:          # SAI: thiếu dấu =
            merged[-1][1] = max(merged[-1][1], c)
        else:
            merged.append([a, c])
    out, prev = [], 0
    for a, c in merged:
        out += [s[prev:a], "<b>", s[a:c], "</b>"]
        prev = c
    out.append(s[prev:])
    return "".join(out)

print("sai :", wrong_merge_strict(["ab", "cd"], "abcd"), "| đúng:", bold_intervals(["ab", "cd"], "abcd"))

sai : <b>ab</b><b>cd</b> | đúng: <b>abcd</b>


### Lỗi 3: Tìm lần xuất hiện bằng nhảy `i += len(w)` → bỏ sót lần chồng nhau

In [19]:
def wrong_skip(words, s):
    bold = [False] * len(s)
    for w in words:
        i = s.find(w)
        while i != -1:
            for k in range(i, i + len(w)):
                bold[k] = True
            i = s.find(w, i + len(w))             # SAI: bỏ qua lần xuất hiện chồng
    return build_from_mask(s, bold)

print("sai :", wrong_skip(["aba"], "ababa"), "| đúng:", bold_one_pass(["aba"], "ababa"))

sai : <b>aba</b>ba | đúng: <b>ababa</b>


`"aba"` xuất hiện ở 0 và 2 (chồng nhau). Nhảy `+len(w)` từ 0 → tìm từ 3 → bỏ sót vị trí 2 → ký tự cuối `a` không đậm.

### Lỗi 4: Chỉ ghi nhận **từ đầu tiên** khớp tại `i` (dùng `break`)

Nếu `words = ["a", "ab"]`, tại `i = 0` khớp `"a"` rồi `break` → bỏ mất `"ab"` dài hơn. Phải lấy `max` qua **mọi** từ khớp.

In [20]:
def wrong_first_match(words, s):
    bold = [False] * len(s)
    end = 0
    for i in range(len(s)):
        for w in words:
            if s.startswith(w, i):
                end = max(end, i + len(w))
                break                              # SAI
        bold[i] = i < end
    return build_from_mask(s, bold)

print("sai :", wrong_first_match(["a", "ab"], "abc"), "| đúng:", bold_one_pass(["a", "ab"], "abc"))

sai : <b>a</b>bc | đúng: <b>ab</b>c


### Lỗi 5: Pha 2 kiểm `bold[i + 1]` khi `i = n - 1` → `IndexError`

Phải kiểm `i == n - 1` **trước** (ngắn mạch `or`): `i == n - 1 or not bold[i + 1]`.

## 14. Mẹo nhận dạng pattern

- "Tô/đánh dấu mọi đoạn khớp, gộp đoạn chồng nhau" → **mảng boolean** hoặc **merge intervals**; nhớ quy ước nửa khoảng `[start, end)` và gộp khi `start <= end_trước`.
- "Duyệt trái sang phải, chỉ cần biết **đã phủ tới đâu**" → một biến `end = max(end, ...)` (giống bài 55 Jump Game, 45 Jump Game II).
- Nhiều từ khóa cần dò tại mỗi vị trí → **Trie** (nâng cao: Aho–Corasick).
- Sinh chuỗi có thẻ/ký tự chèn → gom vào list rồi `''.join`; **không** sửa chuỗi trong lúc còn tìm kiếm trên nó.
- Bài cùng họ: 616 Add Bold Tag in String, 56 Merge Intervals, 57 Insert Interval, 763 Partition Labels, 1065 Index Pairs of a String.

## 15. Biến thể

### 15.1. LeetCode 56 — Merge Intervals (khoảng đóng `[a, b]`)

Với khoảng **đóng**, `[1,4]` và `[4,5]` chồng nhau ở điểm 4 → gộp. Code giống hệt `merge_intervals` — chỉ khác ý nghĩa của "sát nhau".

In [21]:
print(merge_intervals([[1, 3], [2, 6], [8, 10], [15, 18]]))   # [[1, 6], [8, 10], [15, 18]]
print(merge_intervals([[1, 4], [4, 5]]))                       # [[1, 5]]

[[1, 6], [8, 10], [15, 18]]
[[1, 5]]


### 15.2. Chỉ gộp khi **chồng** nhau, giữ riêng khi chỉ sát nhau

Một số hệ thống muốn `"<b>ab</b><b>cd</b>"` thay vì gộp. Đổi điều kiện gộp thành `start < end_trước`.

In [22]:
print(wrong_merge_strict(["ab", "cd"], "abcd"))    # chính là "biến thể" này

<b>ab</b><b>cd</b>


### 15.3. LeetCode 1065 — Index Pairs of a String

Trả mọi cặp `[i, j]` sao cho `s[i..j]` (đóng) là một từ trong `words`, sắp tăng. Trie cho mỗi `i` liệt kê mọi `j` khớp.

In [23]:
def index_pairs(text, words):
    trie = build_trie(words)
    res = []
    for i in range(len(text)):
        node, j = trie, i
        while j < len(text) and text[j] in node:
            node = node[text[j]]
            if "$" in node:
                res.append([i, j])
            j += 1
    return res

print(index_pairs("thestoryofleetcodeandme", ["story", "fleet", "leetcode"]))
print(index_pairs("ababa", ["aba", "ab"]))

[[3, 7], [9, 13], [10, 17]]
[[0, 1], [0, 2], [2, 3], [2, 4]]


## 16. Bài tập

### Bài 1 — Tính tay
`words = ["xy", "yz", "q"]`, `s = "axyzbqq"`. Lập mảng `bold` và viết kết quả.

### Bài 2 — Mọi vị trí xuất hiện (kể cả chồng nhau)
Viết `occurrences(s, w)` trả list mọi `i` mà `s[i:i+len(w)] == w`, **không dùng `find`**.

In [24]:
def occurrences(s: str, w: str) -> list[int]:
    # TODO
    pass

# assert occurrences("aaaa", "aa") == [0, 1, 2]
# assert occurrences("abc", "d") == []

### Bài 3 — Điền chỗ trống Pha 2 (dựng chuỗi từ mảng `bold`)

In [25]:
def exercise_build(s: str, bold: list[bool]) -> str:
    n = len(s)
    res = []
    for i in range(n):
        # TODO 1: mở <b> nếu bold[i] là đầu một dãy True
        res.append(s[i])
        # TODO 2: đóng </b> nếu bold[i] là cuối một dãy True
        pass
    return "".join(res)

# assert exercise_build("aabcd", [False, True, True, True, False]) == "a<b>abc</b>d"
# assert exercise_build("aba", [True, False, True]) == "<b>a</b>b<b>a</b>"

### Bài 4 — Insert Interval (LeetCode 57)
Cho list khoảng **đã gộp và sắp xếp** (nửa khoảng `[a, b)`, gộp khi sát) và một khoảng mới; chèn vào và gộp lại, trong `O(m)` (không sắp xếp lại).

In [26]:
def insert_interval(intervals: list[list[int]], new: list[int]) -> list[list[int]]:
    # TODO
    pass

# assert insert_interval([[1, 3], [6, 9]], [2, 5]) == [[1, 5], [6, 9]]
# assert insert_interval([[1, 2], [3, 5], [6, 7], [8, 10]], [4, 8]) == [[1, 2], [3, 10]]
# assert insert_interval([[1, 2]], [2, 3]) == [[1, 3]]

### Bài 5 — Đếm số ký tự in đậm
Không dựng chuỗi, chỉ trả **số ký tự** của `s` bị in đậm. Dùng ý tưởng biến `end`.

In [27]:
def count_bold_chars(words: list[str], s: str) -> int:
    # TODO
    pass

# assert count_bold_chars(["ab", "bc"], "aabcd") == 3
# assert count_bold_chars(["z"], "abc") == 0

### Bài 6 — Mini challenge: thẻ khác nhau cho từ khác nhau
Mỗi từ khóa có một thẻ riêng: `tags = {"ab": "b", "cd": "i"}`. Các lần xuất hiện **không chồng nhau** (giả định). Bọc mỗi lần xuất hiện bằng thẻ của nó, ví dụ `"<b>ab</b>x<i>cd</i>"`. Nếu hai lần xuất hiện chồng nhau, chọn từ bắt đầu sớm hơn (hòa thì từ dài hơn) và bỏ từ kia.

In [28]:
def tag_words(tags: dict[str, str], s: str) -> str:
    # TODO
    pass

# assert tag_words({"ab": "b", "cd": "i"}, "abxcd") == "<b>ab</b>x<i>cd</i>"
# assert tag_words({"ab": "b", "bc": "i"}, "abc") == "<b>ab</b>c"

## 17. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | Tìm từng từ, đánh dấu | `xy`@1, `yz`@2, `q`@5,6 | bold = `F T T T F T T` |
| 2 | Thử mọi `i` | `range(len(s) - len(w) + 1)` | `if s[i:i+len(w)] == w: res.append(i)` |
| 3 | Đầu dãy / cuối dãy | so với hàng xóm trái/phải | `if bold[i] and (i == 0 or not bold[i-1])` |
| 4 | 3 nhóm: trước, chồng, sau | Duyệt một lượt | thêm khoảng có `b < new[0]`; gộp khi `a <= new[1]`; thêm phần còn lại |
| 5 | Đếm `i < end` | Giống Pha 1 Cách 3 | `cnt += i < end` |
| 6 | Tham lam từ trái | Tại `i`, chọn từ dài nhất khớp | Nhảy `i += len(w)` sau khi bọc; nếu không khớp, thêm `s[i]` |

## 18. Đáp án tham khảo

**Bài 1:** `xy` ở [1,3), `yz` ở [2,4), `q` ở [5,6) và [6,7). Gộp: [1,4) và [5,7) → `bold = F T T T F T T` → `"a<b>xyz</b>b<b>qq</b>"`.

In [29]:
print("Bài 1:", trace_bold(["xy", "yz", "q"], "axyzbqq"))


def answer_occurrences(s, w):
    return [i for i in range(len(s) - len(w) + 1) if s[i:i + len(w)] == w]


def answer_build(s, bold):
    n = len(s)
    res = []
    for i in range(n):
        if bold[i] and (i == 0 or not bold[i - 1]):
            res.append("<b>")
        res.append(s[i])
        if bold[i] and (i == n - 1 or not bold[i + 1]):
            res.append("</b>")
    return "".join(res)


def answer_insert_interval(intervals, new):
    res, i, m = [], 0, len(intervals)
    a, c = new
    while i < m and intervals[i][1] < a:          # kết thúc trước khi new bắt đầu (không sát)
        res.append(intervals[i]); i += 1
    while i < m and intervals[i][0] <= c:         # chồng hoặc sát
        a, c = min(a, intervals[i][0]), max(c, intervals[i][1]); i += 1
    res.append([a, c])
    res.extend(intervals[i:])
    return res


def answer_count_bold(words, s):
    end, cnt = 0, 0
    for i in range(len(s)):
        for w in words:
            if s.startswith(w, i):
                end = max(end, i + len(w))
        cnt += i < end
    return cnt


def answer_tag_words(tags, s):
    out, i = [], 0
    by_len = sorted(tags, key=len, reverse=True)
    while i < len(s):
        for w in by_len:
            if s.startswith(w, i):
                out.append(f"<{tags[w]}>{w}</{tags[w]}>")
                i += len(w)
                break
        else:
            out.append(s[i])
            i += 1
    return "".join(out)


assert answer_occurrences("aaaa", "aa") == [0, 1, 2]
assert answer_occurrences("abc", "d") == []
assert answer_build("aabcd", [False, True, True, True, False]) == "a<b>abc</b>d"
assert answer_build("aba", [True, False, True]) == "<b>a</b>b<b>a</b>"
assert answer_insert_interval([[1, 3], [6, 9]], [2, 5]) == [[1, 5], [6, 9]]
assert answer_insert_interval([[1, 2], [3, 5], [6, 7], [8, 10]], [4, 8]) == [[1, 2], [3, 10]]
assert answer_insert_interval([[1, 2]], [2, 3]) == [[1, 3]]
assert answer_count_bold(["ab", "bc"], "aabcd") == 3
assert answer_count_bold(["z"], "abc") == 0
assert answer_tag_words({"ab": "b", "cd": "i"}, "abxcd") == "<b>ab</b>x<i>cd</i>"
assert answer_tag_words({"ab": "b", "bc": "i"}, "abc") == "<b>ab</b>c"
print("Tất cả đáp án PASS")

words = ['xy', 'yz', 'q'], s = 'axyzbqq'
i=0 s[i]='a' | khớp: []           | end=0 | bold=False
i=1 s[i]='x' | khớp: ['xy']       | end=3 | bold=True
i=2 s[i]='y' | khớp: ['yz']       | end=4 | bold=True
i=3 s[i]='z' | khớp: []           | end=4 | bold=True
i=4 s[i]='b' | khớp: []           | end=4 | bold=False
i=5 s[i]='q' | khớp: ['q']        | end=6 | bold=True
i=6 s[i]='q' | khớp: ['q']        | end=7 | bold=True
  i=1: mở <b> 
  i=3: đóng </b>
  i=5: mở <b> 
  i=6: đóng </b>
=> a<b>xyz</b>b<b>qq</b>
Bài 1: a<b>xyz</b>b<b>qq</b>
Tất cả đáp án PASS


Ghi chú bài 4: với nửa khoảng và quy ước "sát thì gộp", điều kiện "nằm hẳn trước" là `b < new_start` (nếu `b == new_start` thì sát → gộp). Ghi chú bài 6: cú pháp `for ... else` — khối `else` chạy khi vòng `for` **không** bị `break` (tức không từ nào khớp tại `i`).

### Template nhớ nhanh + Checklist submit

```python
class Solution:
    def boldWords(self, words, s):
        n, end = len(s), 0
        bold = [False] * n
        for i in range(n):                         # Pha 1: đánh dấu
            for w in words:
                if s.startswith(w, i):
                    end = max(end, i + len(w))     # phủ xa nhất
            bold[i] = i < end
        res = []
        for i in range(n):                         # Pha 2: dựng chuỗi
            if bold[i] and (i == 0 or not bold[i - 1]):
                res.append("<b>")
            res.append(s[i])
            if bold[i] and (i == n - 1 or not bold[i + 1]):
                res.append("</b>")
        return "".join(res)
```

**Câu thần chú:** *"đánh dấu trên chuỗi gốc — end = max phủ xa nhất — mở ở đầu dãy, đóng ở cuối dãy — sát nhau cũng gộp."*

**Checklist trước khi submit:**
- [ ] Bắt được lần xuất hiện **chồng nhau** (thử mọi `i`, không nhảy `+len(w)`)?
- [ ] Lấy `max` qua **mọi** từ khớp tại `i` (không `break` sớm)?
- [ ] Đoạn **sát nhau** được gộp (`<=` khi merge, hoặc dãy `True` liền mạch)?
- [ ] Không dùng `str.replace` lên chuỗi đã chèn thẻ?
- [ ] Kiểm biên `i == 0` / `i == n - 1` trước khi đọc `bold[i-1]` / `bold[i+1]`?
- [ ] Đã test `words = []`, không khớp, khớp toàn chuỗi?